# 面试问题：3D Gaussian Splatting 的投影与前向 Alpha Compositing 如何实现？

可以直接复述的回答是：每个 3D Gaussian 保存位置、协方差或尺度、颜色和不透明度。相机投影把中心与协方差变到屏幕空间，计算每个像素的二维高斯 alpha。随后必须按深度从前到后排序，以 `w_i=T_i*alpha_i` 累积颜色，并更新透射率 `T_{i+1}=T_i*(1-alpha_i)`。训练用渲染图与目标图的损失反传到 Gaussian 参数，并周期性 densify/prune。错误地相加 alpha 或颠倒深度顺序会在重叠区域产生明显颜色错误。下面用九条射线和三个可训练 Gaussian 手写可微 splatting。

## 真实案例：三个半透明标记在九个屏幕像素上的一维切片

三个 Gaussian 可理解为前景红色路标、中景暗色支架和背景亮色灯牌。为便于逐项打印，本例只渲染相机水平轴的一维切片；深度、颜色和目标像素均为教学构造，不代表真实 3D 场景。

In [1]:
import math  # 导入 logit 初始化需要的数学函数
import warnings  # 导入告警控制模块
import torch  # 导入 PyTorch 张量和自动微分
warnings.filterwarnings("ignore")  # 隐藏环境告警保持输出清晰
torch.set_num_threads(1)  # 固定 CPU 单线程执行
torch.manual_seed(1403)  # 固定参数初始化
rays = torch.linspace(-0.8, 0.8, 9)  # 定义九个屏幕水平像素射线
true_mean_x = torch.tensor([-0.35, 0.15, 0.55])  # 定义三个 Gaussian 世界横坐标
depths = torch.tensor([1.2, 1.8, 2.6])  # 定义从前到后的固定深度
true_scales = torch.tensor([0.24, 0.28, 0.22])  # 定义投影屏幕高斯尺度
true_colors = torch.tensor([0.90, 0.20, 0.75])  # 定义三个灰度颜色
true_opacities = torch.tensor([0.82, 0.70, 0.75])  # 定义三个中心不透明度
scene_labels = ["前景路标", "中景支架", "背景灯牌"]  # 定义三个高斯业务语义
print("输入预览：gaussian | mean_x | depth | scale | color | opacity")  # 输出场景参数表头
for index, label in enumerate(scene_labels):  # 遍历三个高斯点元
    print(f"{label:6} | {true_mean_x[index]:.3f} | {depths[index]:.2f} | {true_scales[index]:.3f} | {true_colors[index]:.2f} | {true_opacities[index]:.2f}")  # 展示可读参数
print("九条屏幕射线：", rays.tolist())  # 展示至少六个真实渲染样本

输入预览：gaussian | mean_x | depth | scale | color | opacity
前景路标   | -0.350 | 1.20 | 0.240 | 0.90 | 0.82
中景支架   | 0.150 | 1.80 | 0.280 | 0.20 | 0.70
背景灯牌   | 0.550 | 2.60 | 0.220 | 0.75 | 0.75
九条屏幕射线： [-0.800000011920929, -0.6000000238418579, -0.4000000059604645, -0.20000000298023224, 0.0, 0.20000000298023224, 0.4000000059604645, 0.6000000238418579, 0.800000011920929]


## Baseline / 基线：忽略遮挡直接把 Gaussian 颜色相加

基线使用真实 Gaussian 参数，却把所有 `alpha*color` 直接相加并截断到 1。它没有透射率，因此重叠像素会过亮。

In [2]:
def alpha_matrix(ray_positions, mean_x, scales, opacities):  # 计算所有射线到所有 Gaussian 的二维简化 alpha
    projected_mean = mean_x / depths  # 用针孔模型近似把世界横坐标投影到屏幕
    normalized_distance = (ray_positions[:, None] - projected_mean[None, :]) / scales[None, :]  # 计算每条射线的标准化屏幕距离
    return opacities[None, :] * torch.exp(-0.5 * normalized_distance.square())  # 形成九乘三高斯 alpha 矩阵
true_alpha = alpha_matrix(rays, true_mean_x, true_scales, true_opacities)  # 计算权威场景每像素 alpha
additive_baseline = torch.clamp((true_alpha * true_colors[None, :]).sum(dim=1), 0.0, 1.0)  # 错误地忽略遮挡直接求和
print("ray | alpha(front,mid,back) | additive_color")  # 输出相加基线中间量表头
for ray_index, ray in enumerate(rays):  # 遍历九个屏幕像素
    print(f"{ray:+.2f} | {[round(value, 3) for value in true_alpha[ray_index].tolist()]} | {additive_baseline[ray_index]:.3f}")  # 展示重叠 alpha 与过亮颜色

ray | alpha(front,mid,back) | additive_color
-0.80 | [0.087, 0.005, 0.0] | 0.079
-0.60 | [0.359, 0.036, 0.001] | 0.331
-0.40 | [0.741, 0.158, 0.016] | 0.710
-0.20 | [0.762, 0.42, 0.13] | 0.868
+0.00 | [0.392, 0.67, 0.472] | 0.841
+0.20 | [0.101, 0.642, 0.749] | 0.781
+0.40 | [0.013, 0.369, 0.52] | 0.475
+0.60 | [0.001, 0.128, 0.158] | 0.145
+0.80 | [0.0, 0.026, 0.021] | 0.021


## 核心实现：可微投影、深度排序与前向 compositing

模型学习横坐标、log-scale、color logit 和 opacity logit；深度作为已知相机排序 buffer。forward 返回颜色、alpha 和实际贡献权重。

In [3]:
def inverse_sigmoid(value):  # 把零到一参数转换为可训练 logit
    return torch.log(value / (1.0 - value))  # 返回 sigmoid 的解析逆变换
class GaussianSlice(torch.nn.Module):  # 定义三个可训练 Gaussian 的一维切片渲染器
    def __init__(self, mean_x, scales, colors, opacities):  # 初始化可训练 splat 参数
        super().__init__()  # 初始化 PyTorch 模块基类
        self.mean_x = torch.nn.Parameter(mean_x.clone())  # 注册世界横坐标
        self.log_scales = torch.nn.Parameter(torch.log(scales.clone()))  # 用对数参数保证尺度为正
        self.color_logits = torch.nn.Parameter(inverse_sigmoid(colors.clone()))  # 用 logit 参数保证颜色在零到一
        self.opacity_logits = torch.nn.Parameter(inverse_sigmoid(opacities.clone()))  # 用 logit 参数保证不透明度在零到一
        self.register_buffer("depths", depths.clone())  # 保存不训练的固定深度
    def forward(self, ray_positions, reverse_depth=False):  # 执行可微 splatting 和 alpha compositing
        scales = torch.exp(self.log_scales)  # 把 log-scale 转为正屏幕尺度
        colors = torch.sigmoid(self.color_logits)  # 把颜色 logits 映射到合法范围
        opacities = torch.sigmoid(self.opacity_logits)  # 把 opacity logits 映射到合法范围
        projected_mean = self.mean_x / self.depths  # 投影三个世界中心到屏幕坐标
        normalized_distance = (ray_positions[:, None] - projected_mean[None, :]) / scales[None, :]  # 计算九射线到三个投影中心距离
        alpha = opacities[None, :] * torch.exp(-0.5 * normalized_distance.square())  # 计算九乘三像素不透明度
        order = torch.argsort(self.depths, descending=reverse_depth)  # 正确或故意反向排列深度
        transmittance = torch.ones(len(ray_positions))  # 初始化每条射线尚未遮挡的透射率
        rendered = torch.zeros(len(ray_positions))  # 初始化渲染灰度颜色
        contribution_columns = []  # 收集每个排序 Gaussian 的实际权重
        for gaussian_index in order:  # 按深度依次执行 front-to-back 合成
            contribution = transmittance * alpha[:, gaussian_index]  # 计算当前高斯权重 T*alpha
            rendered = rendered + contribution * colors[gaussian_index]  # 累加当前高斯颜色贡献
            contribution_columns.append(contribution)  # 保存逐射线贡献供解释
            transmittance = transmittance * (1.0 - alpha[:, gaussian_index])  # 更新剩余透射率
        contributions = torch.stack(contribution_columns, dim=1)  # 拼接排序后的九乘三贡献矩阵
        return rendered, alpha, contributions, transmittance  # 返回颜色、alpha、权重和剩余透射率
true_scene = GaussianSlice(true_mean_x, true_scales, true_colors, true_opacities)  # 创建生成监督图的权威场景
for parameter in true_scene.parameters():  # 遍历权威场景参数
    parameter.requires_grad_(False)  # 冻结目标场景防止训练污染
with torch.no_grad():  # 关闭目标渲染计算图
    target_pixels, target_alpha, target_contributions, target_transmittance = true_scene(rays)  # 生成九个权威像素
baseline_mse = float(((additive_baseline - target_pixels) ** 2).mean())  # 计算相加基线与真实遮挡渲染误差
center_index = len(rays) // 2  # 选择重叠最明显的中心射线
print("中心射线 alpha：", target_alpha[center_index].tolist())  # 展示三个高斯覆盖程度
print("中心射线贡献 T*alpha：", target_contributions[center_index].tolist())  # 展示遮挡后的真实权重
print(f"中心 target={target_pixels[center_index]:.4f}，additive={additive_baseline[center_index]:.4f}，baseline MSE={baseline_mse:.6f}")  # 对比错误相加与正确合成

中心射线 alpha： [0.39184024930000305, 0.6696745157241821, 0.47238537669181824]
中心射线贡献 T*alpha： [0.39184024930000305, 0.40726912021636963, 0.0948978140950203]
中心 target=0.5053，additive=0.8409，baseline MSE=0.034755


## 真实训练：从扰动 Gaussian 参数拟合九条射线

训练使用手写 Adam 状态更新，不调用渲染库。输出损失、梯度和 Gaussian 参数轨迹。

In [4]:
train_scene = GaussianSlice(true_mean_x + torch.tensor([0.12, -0.10, 0.08]), true_scales * 1.25, torch.tensor([0.65, 0.40, 0.55]), torch.tensor([0.60, 0.55, 0.60]))  # 创建扰动初始场景
first_moment = {name: torch.zeros_like(parameter) for name, parameter in train_scene.named_parameters()}  # 初始化手写 Adam 一阶矩
second_moment = {name: torch.zeros_like(parameter) for name, parameter in train_scene.named_parameters()}  # 初始化手写 Adam 二阶矩
training_trace = []  # 保存关键步损失和可解释参数
for step in range(1, 401):  # 对九像素监督执行四百步优化
    prediction, alpha, contributions, transmittance = train_scene(rays)  # 真实执行可微 Gaussian forward
    loss = ((prediction - target_pixels) ** 2).mean()  # 计算渲染像素均方误差
    loss.backward()  # 真实执行 backward 到位置、尺度、颜色和不透明度
    gradient_norm = torch.sqrt(sum(parameter.grad.square().sum() for parameter in train_scene.parameters()))  # 计算全局 Gaussian 梯度范数
    with torch.no_grad():  # 关闭手写 Adam 更新计算图
        for name, parameter in train_scene.named_parameters():  # 遍历四组可训练 Gaussian 参数
            first_moment[name] = 0.9 * first_moment[name] + 0.1 * parameter.grad  # 更新一阶矩
            second_moment[name] = 0.999 * second_moment[name] + 0.001 * parameter.grad.square()  # 更新二阶矩
            corrected_first = first_moment[name] / (1.0 - 0.9 ** step)  # 对一阶矩做偏差修正
            corrected_second = second_moment[name] / (1.0 - 0.999 ** step)  # 对二阶矩做偏差修正
            parameter -= 0.03 * corrected_first / (corrected_second.sqrt() + 1e-8)  # 应用手写 Adam 参数更新
            parameter.grad.zero_()  # 清空本步梯度
    if step in {1, 10, 100, 400}:  # 保存关键训练节点
        training_trace.append((step, float(loss), float(gradient_norm), train_scene.mean_x.detach().clone(), torch.exp(train_scene.log_scales.detach()).clone()))  # 记录位置和尺度轨迹
with torch.no_grad():  # 进入训练后渲染评估阶段
    trained_pixels, trained_alpha, trained_contributions, trained_transmittance = train_scene(rays)  # 渲染九个训练后像素
trained_mse = float(((trained_pixels - target_pixels) ** 2).mean())  # 计算可微 splatting 主方案误差
print("step | MSE | grad_norm | mean_x | scales")  # 输出 Gaussian 训练轨迹表头
for item in training_trace:  # 遍历四个关键优化节点
    print(f"{item[0]:4d} | {item[1]:.7f} | {item[2]:.6f} | {[round(value, 3) for value in item[3].tolist()]} | {[round(value, 3) for value in item[4].tolist()]}")  # 展示参数真实更新
print("ray | target | additive_baseline | trained")  # 输出九射线结果表头
for ray_index, ray in enumerate(rays):  # 遍历九个屏幕像素
    print(f"{ray:+.2f} | {target_pixels[ray_index]:.4f} | {additive_baseline[ray_index]:.4f} | {trained_pixels[ray_index]:.4f}")  # 展示逐像素基线和学习结果
print(f"MSE：additive={baseline_mse:.6f}，trained splatting={trained_mse:.6f}")  # 汇总同射线对照

step | MSE | grad_norm | mean_x | scales
   1 | 0.0198825 | 0.063704 | [-0.26, 0.02, 0.6] | [0.309, 0.361, 0.283]
  10 | 0.0077180 | 0.029781 | [-0.383, -0.199, 0.35] | [0.323, 0.386, 0.321]
 100 | 0.0001177 | 0.000614 | [-0.336, -0.504, 0.624] | [0.183, 0.277, 0.255]
 400 | 0.0000001 | 0.000001 | [-0.349, -0.361, 0.814] | [0.197, 0.298, 0.203]
ray | target | additive_baseline | trained
-0.80 | 0.0792 | 0.0793 | 0.0791
-0.60 | 0.3283 | 0.3311 | 0.3284
-0.40 | 0.6773 | 0.7099 | 0.6772
-0.20 | 0.7195 | 0.8678 | 0.7196
+0.00 | 0.5053 | 0.8409 | 0.5052
+0.20 | 0.3869 | 0.7806 | 0.3871
+0.40 | 0.3272 | 0.4752 | 0.3270
+0.60 | 0.1294 | 0.1446 | 0.1298
+0.80 | 0.0206 | 0.0210 | 0.0199
MSE：additive=0.034755，trained splatting=0.000000


## 失败案例与修正：深度顺序颠倒

Alpha compositing 不可交换。使用完全相同的真实 Gaussian，仅把排序从近到远改成远到近，重叠区域颜色就会变化；恢复 front-to-back 后与目标逐值一致。

In [5]:
with torch.no_grad():  # 关闭深度顺序对照的梯度记录
    reversed_pixels, reversed_alpha, reversed_contributions, reversed_transmittance = true_scene(rays, reverse_depth=True)  # 故意从远到近合成
    fixed_pixels, fixed_alpha, fixed_contributions, fixed_transmittance = true_scene(rays, reverse_depth=False)  # 使用正确近到远合成
reversed_mse = float(((reversed_pixels - target_pixels) ** 2).mean())  # 计算错误深度顺序渲染误差
fixed_mse = float(((fixed_pixels - target_pixels) ** 2).mean())  # 计算正确顺序重放误差
maximum_order_error = float((reversed_pixels - target_pixels).abs().max())  # 找到顺序错误最大像素偏差
print("中心射线 reversed contributions：", reversed_contributions[center_index].tolist())  # 展示远到近贡献权重
print("中心射线 fixed contributions：", fixed_contributions[center_index].tolist())  # 展示近到远贡献权重
print(f"中心颜色：reversed={reversed_pixels[center_index]:.4f}，fixed={fixed_pixels[center_index]:.4f}，target={target_pixels[center_index]:.4f}")  # 展示重叠颜色错误
print(f"order MSE：reversed={reversed_mse:.6f}，fixed={fixed_mse:.6f}，max_error={maximum_order_error:.4f}")  # 汇总深度排序修正

中心射线 reversed contributions： [0.47238537669181824, 0.3533300459384918, 0.06829170137643814]
中心射线 fixed contributions： [0.39184024930000305, 0.40726912021636963, 0.0948978140950203]
中心颜色：reversed=0.4864，fixed=0.5053，target=0.5053
order MSE：reversed=0.011650，fixed=0.000000，max_error=0.2151


## 结果解读

`alpha` 只表示覆盖强度，实际颜色权重是乘上之前所有 Gaussian 剩余透射率后的 `T*alpha`。相加基线即使拿到真实参数仍过亮；训练模型经过真实 forward/backward 后拟合九条射线。深度反例说明 compositing 顺序是渲染方程的一部分，而不是无关实现细节。

## 生产边界

本例是一维灰度切片，深度固定，也没有完整三维协方差、球谐颜色、相机姿态、tile rasterizer、densification 或 pruning。生产 3DGS 依赖 CUDA 排序与梯度 kernel，并需多视角图像、相机标定、曝光处理和内存控制。质量应报告 PSNR/SSIM/LPIPS 与真实 FPS，不能用九射线拟合误差外推。

## 最小回归测试

In [6]:
assert len(rays) >= 6 and target_alpha.shape == (len(rays), 3)  # 保证案例包含多条射线和三个高斯
assert torch.all((target_alpha >= 0.0) & (target_alpha <= 1.0))  # 保证高斯 alpha 落在合法范围
assert training_trace[-1][1] < training_trace[0][1]  # 保证真实 backward 优化降低渲染损失
assert trained_mse < baseline_mse  # 保证可微 splatting 拟合优于错误相加基线
assert reversed_mse > 0.0 and maximum_order_error > 0.01  # 保证深度颠倒失败真实产生像素误差
assert fixed_mse == 0.0 and torch.equal(fixed_pixels, target_pixels)  # 保证恢复近到远顺序逐值重放目标
assert torch.isfinite(trained_pixels).all()  # 保证学习后的九像素输出数值稳定